In [ ]:
import pandas as pd
import requests
import time
import json
import os
import re
from datetime import datetime


def map_action_numbers(old_df, new_df):
    """
    Map actionNumber from new dataset to old dataset based on period and time information.
    
    Parameters:
    -----------
    old_df : pandas.DataFrame
        The old NBA play-by-play dataset
    new_df : pandas.DataFrame
        The new NBA play-by-play dataset with actionNumber
        
    Returns:
    --------
    pandas.DataFrame
        Old dataset with mapped actionNumber from new dataset
    """
    # Make a copy of the old DataFrame to avoid modifying the original
    old_df_with_action = old_df.copy()
    old_df_with_action['actionNumber'] = None

    # Function to convert "PT12M00.00S" format to seconds remaining in period
    def clock_to_seconds(clock_str):
        if pd.isna(clock_str) or clock_str is None:
            return None
        match = re.match(r'PT(\d+)M(\d+\.\d+)S', clock_str)
        if match:
            minutes = int(match.group(1))
            seconds = float(match.group(2))
            return minutes * 60 + seconds
        return None

    teamid = old_df['TEAM_ID'].iloc[0]
    new_df = new_df[new_df['teamId'] == teamid]
    new_df['seconds_remaining'] = new_df['clock'].apply(clock_to_seconds)

    # Convert MM:SS format to seconds from start of the period
    def mmss_to_seconds_from_start(time_str, period):
        if pd.isna(time_str) or time_str is None:
            return None

        parts = time_str.split(':')
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return 720 - (minutes * 60 + seconds)
        return None

    old_df_with_action['start_seconds_in_period'] = old_df_with_action.apply(
        lambda row: mmss_to_seconds_from_start(row['STARTTIME'], row['PERIOD']),
        axis=1
    )

    old_df_with_action['end_seconds_in_period'] = old_df_with_action.apply(
        lambda row: mmss_to_seconds_from_start(row['ENDTIME'], row['PERIOD']),
        axis=1
    )

    # Iterate through each row in the old dataset
    for idx, old_row in old_df_with_action.iterrows():
        period_matches = new_df[new_df['period'] == old_row['PERIOD']]

        if len(period_matches) == 0:
            continue

        time_matches = period_matches[
            (period_matches['seconds_remaining'] >= old_row['start_seconds_in_period']) &
            (period_matches['seconds_remaining'] <= old_row['end_seconds_in_period'])
        ]

        if len(time_matches) > 0:
            old_df_with_action.at[idx, 'actionNumber'] = time_matches['actionNumber'].iloc[0]

    return old_df_with_action


def advanced_map_action_numbers(old_df, new_df):
    """
    A more sophisticated mapping that tries to match events based on descriptions
    in addition to timing information.
    
    Parameters:
    -----------
    old_df : pandas.DataFrame
        The old NBA play-by-play dataset
    new_df : pandas.DataFrame
        The new NBA play-by-play dataset with actionNumber
        
    Returns:
    --------
    pandas.DataFrame
        Old dataset with mapped actionNumber from new dataset
    """
    teamid = old_df['TEAM_ID'].iloc[0]
    new_df = new_df[new_df['teamId'] == teamid]
    old_df_with_action = map_action_numbers(old_df, new_df)

    for idx, old_row in old_df_with_action.iterrows():
        if not pd.isna(old_row['actionNumber']):
            continue

        period_matches = new_df[new_df['period'] == old_row['PERIOD']]
        old_desc = old_row['DESCRIPTION'].lower() if not pd.isna(old_row['DESCRIPTION']) else ""

        for _, new_row in period_matches.iterrows():
            new_desc = new_row['description'].lower() if not pd.isna(new_row['description']) else ""

            if old_desc and new_desc:
                common_words = set(old_desc.split()).intersection(set(new_desc.split()))
                if len(common_words) >= 2:
                    old_df_with_action.at[idx, 'actionNumber'] = new_row['actionNumber']
                    break

    return old_df_with_action


def match_by_game_events(old_df, new_df):
    """
    Additional approach: try to align events sequence by sequence within games.
    
    Parameters:
    -----------
    old_df : pandas.DataFrame
        The old NBA play-by-play dataset
    new_df : pandas.DataFrame
        The new NBA play-by-play dataset with actionNumber
        
    Returns:
    --------
    pandas.DataFrame
        Old dataset with mapped actionNumber from new dataset
    """
    old_games = old_df.groupby('GAMEID')
    teamid = old_df['TEAM_ID'].iloc[0]

    result_df = old_df.copy()
    result_df['actionNumber'] = None

    for game_id, old_game_df in old_games:
        new_game_df = new_df[(new_df['game_id'] == game_id) & (new_df['teamId'] == teamid)]

        if len(new_game_df) == 0:
            continue

        old_game_df = old_game_df.sort_values(['PERIOD', 'start_seconds_in_period'])
        new_game_df = new_game_df.sort_values(['period', 'timeActual'])

        for idx, old_row in old_game_df.iterrows():
            period_matches = new_game_df[new_game_df['period'] == old_row['PERIOD']]
            if len(period_matches) > 0:
                action_num = period_matches['actionNumber'].iloc[0]
                result_df.loc[idx, 'actionNumber'] = action_num
                new_game_df = new_game_df[new_game_df['actionNumber'] != action_num]

    return result_df


def get_action_numbers(old_df, new_df):
    """
    Map actionNumber from the new dataset to the old dataset using multiple approaches.
    
    Parameters:
    -----------
    old_df : pandas.DataFrame
        Old NBA play-by-play dataset
    new_df : pandas.DataFrame
        New NBA play-by-play dataset with actionNumber
        
    Returns:
    --------
    pandas.DataFrame
        Old dataset with mapped actionNumber from new dataset
    """
    teamid = old_df['TEAM_ID'].iloc[0]
    new_df = new_df[new_df['teamId'] == teamid]

    result_df = map_action_numbers(old_df, new_df)
    mapped_count = result_df['actionNumber'].notna().sum()
    total_count = len(result_df)

    print(f"Time-based mapping: {mapped_count}/{total_count} rows mapped ({mapped_count/total_count:.1%})")
 
    return result_df

def pull_data(url):
    headers = {
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Referer": "https://stats.nba.com/",
        "Origin": "https://stats.nba.com",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
    }

    response = requests.get(url, headers=headers)
    json = response.json()
    
    # Handle the video events format
    if 'resultSets' in json and isinstance(json['resultSets'], dict):
        if 'Meta' in json['resultSets'] and 'videoUrls' in json['resultSets']['Meta']:
            video_urls = json['resultSets']['Meta']['videoUrls']
            playlist = json['resultSets'].get('playlist', [])
            
            # Convert video URLs to dataframe
            video_df = pd.DataFrame(video_urls)
            
            # Convert playlist to dataframe
            playlist_df = pd.DataFrame(playlist)
            
            # Merge video and playlist data
            if not playlist_df.empty:
                df = pd.concat([video_df, playlist_df], axis=1)
            else:
                df = video_df

        else:
            # Fallback in case no video data is found
            df = pd.DataFrame()

    # Handle the original stats format
    elif 'resultSets' in json and isinstance(json['resultSets'], list):
        if len(json["resultSets"]) == 1:
            data = json["resultSets"][0]["rowSet"]
            columns = json["resultSets"][0]["headers"]
            df = pd.DataFrame.from_records(data, columns=columns)
        else:
            data = json["resultSets"][1]["rowSet"]
            columns = json["resultSets"][1]["headers"]["columnNames"]
            df = pd.DataFrame.from_records(data, columns=columns)

    else:
        # Empty dataframe if no recognizable format is found
        df = pd.DataFrame()

    time.sleep(1.2)
    return df

import pandas as pd



# List of NBA team acronyms


# Example: Access the DataFrame for the Atlanta Hawks
# atl_df = team_dfs['ATL']

result_frames=[]
teams = [
    'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 
    'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]

# Dictionary to store DataFrames for each team
team_dfs = {}

# Loop through each team and read the CSV file
for team in teams:
    file_path = f'2025/{team}_2025_clips_with_players.csv'
    
    if os.path.exists(file_path):  # Ensure the file exists before reading
        team_dfs[team] = pd.read_csv(file_path)
        print(f"Loaded {team}")
    else:
        print(f"File not found: {file_path}")


    all_df = pd.read_csv(file_path)

    # Identify GAME_IDs with at least one non-NaN URL
    valid_games = all_df[all_df['URL'].notna()]['GAMEID'].unique()

    # Filter out GAME_IDs without any non-NaN URLs
    missing_url_games = all_df[~all_df['GAMEID'].isin(valid_games)]['GAMEID'].unique()

    print("GAME_IDs without at least one non-NaN URL:")
    print(missing_url_games)


    missing=all_df[all_df.GAMEID.isin(missing_url_games)]
    missing

    # API endpoint

    all_rows = []
    for game_id in missing_url_games:
  


    # Direct API endpoint for play-by-play data

        url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_00{game_id}.json"

        # Set headers to mimic a browser request
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        }

        # Fetch the JSON data
        response = requests.get(url, headers=headers)


        if response.status_code == 200:
            data = response.json()

            actions = data.get('game', {}).get('actions', [])
                
                # Convert each action into a dictionary and add to the list
            for action in actions:
                action['game_id'] = game_id  # Add the game ID as a column
                all_rows.append(action)
            
        else:
            print(f"Failed to fetch data: {response.status_code}")
        time.sleep(1)
    time.sleep(1)
    teamdf = pd.DataFrame(all_rows)


    old_df=missing.copy()

    new_df=teamdf.copy()
    print(new_df.columns)

    # Example usage
    old_df['GAMEID']='00'+old_df['GAMEID'].astype(str)
    old_df.sort_values(by='GAMEDATE',inplace=True)
    new_df.sort_values(by='timeActual',inplace=True)
    print(team)

    result = get_action_numbers(old_df=old_df,new_df=new_df)
    result.sort_values(by=['GAMEDATE','PERIOD','start_seconds'],inplace=True)


    # Example DataFrame
    print(result[['actionNumber','DESCRIPTION','PERIOD', 'start_seconds', 'end_seconds',]])
    result_frames.append(result)

Loaded ATL
GAME_IDs without at least one non-NaN URL:
[22400239 22400719 22400945 22400960 22400978 22400993]
Index(['actionNumber', 'clock', 'timeActual', 'period', 'periodType',
       'actionType', 'subType', 'qualifiers', 'personId', 'x', 'y',
       'possession', 'scoreHome', 'scoreAway', 'edited', 'orderNumber',
       'isTargetScoreLastPeriod', 'xLegacy', 'yLegacy', 'isFieldGoal', 'side',
       'description', 'personIdsFilter', 'game_id', 'teamId', 'teamTricode',
       'descriptor', 'jumpBallRecoveredName', 'jumpBallRecoverdPersonId',
       'playerName', 'playerNameI', 'jumpBallWonPlayerName',
       'jumpBallWonPersonId', 'jumpBallLostPlayerName', 'jumpBallLostPersonId',
       'area', 'areaDetail', 'shotDistance', 'shotResult', 'blockPlayerName',
       'blockPersonId', 'shotActionNumber', 'reboundTotal',
       'reboundDefensiveTotal', 'reboundOffensiveTotal', 'pointsTotal',
       'assistPlayerNameInitial', 'assistPersonId', 'assistTotal',
       'officialId', 'turnoverTo

In [63]:
all_missing=pd.concat(result_frames)
all_missing

,index,ENDTIME,EVENTS,FG2A,FG2M,FG3A,FG3M,GAMEDATE,GAMEID,NONSHOOTINGFOULSTHATRESULTEDINFTS,...,Year,start_seconds,end_seconds,mid_seconds,players_on,opp_players_on,season,actionNumber,start_seconds_in_period,end_seconds_in_period
3097,111322,11:34,Clingan BLOCK (1 BLK): MISS Capela 9' Hook Sho...,1,0,0,0,2024-11-17,0022400239,0,...,2024,0,26,13.0,203991|1629027|1630552|1630700|1642258,203924|1630703|1631101|1641739|1642270,2024-25,179,0,26
3108,111321,11:34,Clingan BLOCK (1 BLK): MISS Capela 9' Hook Sho...,1,0,0,0,2024-11-17,0022400239,0,...,2024,0,26,13.0,203991|1629027|1630552|1630700|1642258,203924|1630703|1631101|1641739|1642270,2024-25,179,0,26
3099,111320,11:23,Young 25' 3PT Running Jump Shot (3 PTS) (Risac...,0,0,1,1,2024-11-17,0022400239,0,...,2024,32,37,34.5,203991|1629027|1630552|1630700|1642258,203924|1630703|1631101|1641739|1642270,2024-25,152,32,37
3100,111319,10:54,Young Out of Bounds - Bad Pass Turnover Turnov...,0,0,0,0,2024-11-17,0022400239,0,...,2024,63,66,64.5,203991|1629027|1630552|1630700|1642258,203924|1630703|1631101|1641739|1642270,2024-25,174,63,66
3098,111318,10:35,Risacher 1' Running Dunk (2 PTS) (Johnson 1 AS...,1,1,0,0,2024-11-17,0022400239,0,...,2024,82,85,83.5,203991|1629027|1630552|1630700|1642258,203924|1630703|1631101|1641739|1642270,2024-25,171,82,85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8001,394776,01:16,MISS Johnson 3PT Jump Shot\nJAZZ Rebound\n,0,0,1,0,2025-03-19,0022400627,0,...,2025,2785,2804,2794.5,1630264|1630550|1641732|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,531,625,644
8004,394773,00:52,MISS Vukcevic 25' 3PT Jump Shot\nGeorge REBOUN...,0,0,1,0,2025-03-19,0022400627,0,...,2025,2816,2828,2822.0,1630550|1641732|1641774|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,529,656,668
8003,394774,00:52,MISS Vukcevic 25' 3PT Jump Shot\nGeorge REBOUN...,0,0,1,0,2025-03-19,0022400627,0,...,2025,2816,2828,2822.0,1630550|1641732|1641774|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,529,656,668
7985,394772,00:18,Filipowski P.FOUL (P3.T4) (N.Buchert)\nVukcevi...,1,1,0,0,2025-03-19,0022400627,0,...,2025,2846,2862,2854.0,1630550|1641732|1641774|1641798|1642358,1630548|1641718|1642262|1642268|1642271,2024-25,525,686,702


In [64]:

# result = pd.DataFrame({'actionNumber': [1, 2, 3, None, 5, None, 7], 'DESCRIPTION': ['A', 'B', 'C', 'D', 'E', 'F', 'G']})
